# AI Diary : Notebook 03: Final Model Training and Evaluation

**What this notebook does:**
This is the main notebook for the project. It picks up where notebook 02 left off and runs the full training pipeline from start to finish.

1. Loads all three dataset splits (train, dev, test) from GoEmotions
2. Computes class weights to handle the label imbalance found in the EDA
3. Fine-tunes RoBERTa on the full training set across 3 epochs
4. Evaluates the model properly on the held-out test set
5. Produces all charts and figures needed for the report
6. Loads survey responses and runs them through the model
7. Compares model predictions against what participants said they felt
8. Generates empathetic reflections using the template system


In [ ]:
# Cell 1
# ======================================================
# Imports

# System utilities
import os
import glob
from zipfile import ZipFile
from collections import Counter

# Data handling
import pandas as pd
import numpy as np

# Charts
import matplotlib.pyplot as plt
import seaborn as sns

# Model evaluation
from sklearn.metrics import (
    f1_score,
    classification_report,
    confusion_matrix
)

# Deep learning
import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

# HuggingFace dataset helper
from datasets import Dataset

# Colab file uploader
from google.colab import files
# Turn off external logging tools (Weights & Biases etc)
os.environ["WANDB_DISABLED"] = "true"



In [ ]:
# Cell 2
# ======================================================
# Upload and Extract the GoEmotions Dataset

uploaded = files.upload()

zip_path = list(uploaded.keys())[0]
os.makedirs("goemotions", exist_ok=True)

with ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("goemotions")



In [ ]:
# Cell 3
# ======================================================
# Load Train, Dev, and Test Splits

def find_file(pattern):
    matches = [
        f for f in glob.glob(pattern, recursive=True)
        if not os.path.basename(f).startswith("._")
    ]
    if not matches:
        raise FileNotFoundError(f"Could not find: {pattern}")
    return matches[0]

train_path = find_file("goemotions/**/train.tsv")
dev_path   = find_file("goemotions/**/dev.tsv")
test_path  = find_file("goemotions/**/test.tsv")
emo_path   = find_file("goemotions/**/emotions.txt")

col_names = ["text", "labels", "id"]

train_df = pd.read_csv(train_path, sep="\t", names=col_names)
dev_df   = pd.read_csv(dev_path,   sep="\t", names=col_names)
test_df  = pd.read_csv(test_path,  sep="\t", names=col_names)

with open(emo_path, encoding="utf-8") as f:
    emotions = [line.strip() for line in f if line.strip()]

NUM_EMOTIONS = len(emotions)

print("Splits loaded:")
print(" Train    :", len(train_df))
print(" Dev      :", len(dev_df))
print(" Test     :", len(test_df))
print(" Emotions :", NUM_EMOTIONS)


In [ ]:
# Cell 4
# ======================================================
# Dataset Overview

print("Training set structure")
train_df.info()

print("5 rows of the training set")
print(train_df.head())

print("Summary statistics")
print(train_df.describe(include="all"))


In [ ]:
# Cell 5
# ======================================================
# Convert Emotion Labels into Binary Vectors


def encode_labels(label_string):
    vector = np.zeros(NUM_EMOTIONS)
    for label in str(label_string).split(","):
        if label.strip().isdigit():
            vector[int(label.strip())] = 1
    return vector

train_df["label_vector"] = train_df["labels"].apply(encode_labels)
dev_df["label_vector"]   = dev_df["labels"].apply(encode_labels)
test_df["label_vector"]  = test_df["labels"].apply(encode_labels)

# example
example_text   = train_df["text"].iloc[0]
example_labels = train_df["labels"].iloc[0]
example_vector = train_df["label_vector"].iloc[0]

print("Example text    :", example_text)
print("Raw label string:", example_labels)
print("Encoded vector  :", example_vector)
print("Active emotions :", [emotions[i] for i, v in enumerate(example_vector) if v == 1])


In [ ]:
# Cell 6
# ======================================================
# Compute Class Weights to Handle Label Imbalance

label_matrix = np.stack(train_df["label_vector"].values)
label_counts = label_matrix.sum(axis=0)

# Tiny epsilon added to avoid dividing by zero for any class with 0 samples
epsilon = 1e-6
class_weights = len(train_df) / (NUM_EMOTIONS * (label_counts + epsilon))
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

print("Class weights computed.")
print(f"Lowest weight  : {class_weights.min():.4f}  - {emotions[class_weights.argmin()]} (most common)")
print(f"Highest weight : {class_weights.max():.4f}  - {emotions[class_weights.argmax()]} (rarest)")
print()
print("All emotions and their weights:")
for emo, count, w in zip(emotions, label_counts, class_weights):
    print(f"  {emo:20s}  count: {int(count):5d}   weight: {w:.4f}")


In [ ]:
# Cell 7
# ======================================================
# Tokenise the Text for RoBERTa

MODEL_NAME = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )


def build_dataset(df):
    ds = Dataset.from_dict({
        "text":   df["text"].astype(str).tolist(),
        "labels": np.stack(df["label_vector"].values).tolist()
    })
    ds = ds.map(tokenize, batched=True)
    ds.set_format("torch")
    return ds

print("Building tokenised train dataset.")
train_ds = build_dataset(train_df)
print("Building tokenised dev dataset.")
dev_ds = build_dataset(dev_df)
print("Building tokenised test dataset.")
test_ds = build_dataset(test_df)



In [ ]:
# Cell 8
# ======================================================
# Define Evaluation Metrics

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = torch.sigmoid(torch.tensor(logits)).numpy()

    predictions = (probs >= 0.3).astype(int)

    return {
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
        "micro_f1": f1_score(labels, predictions, average="micro", zero_division=0),
    }


In [ ]:
# Cell 9
# ======================================================
# Custom Trainer with Weighted Loss



class WeightedTrainer(Trainer):

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits

        weights = class_weights_tensor.to(logits.device)

        loss_fn = nn.BCEWithLogitsLoss(pos_weight=weights)
        loss    = loss_fn(logits, labels.float())

        return (loss, outputs) if return_outputs else loss

In [ ]:
# Cell 10
# ======================================================
# Load the RoBERTa Model

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_EMOTIONS,
    problem_type="multi_label_classification"
)



In [ ]:
# Cell 11
# ======================================================
# Training Arguments

training_args = TrainingArguments(
    output_dir="roberta_final",

    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",

    logging_steps=50,
    report_to="none",

    optim="adamw_torch"
)

print("Training arguments set.")
print(f"  Epochs     : {training_args.num_train_epochs}")
print(f"  Batch size : {training_args.per_device_train_batch_size}")
print(f"  LR         : {training_args.learning_rate}")
print(f"  Optimiser  : {training_args.optim}")


In [ ]:
# Cell 12
# ======================================================
# Initialise the Trainer

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    compute_metrics=compute_metrics
)

In [ ]:
# Cell 13
# ======================================================
# Train the Model

train_result = trainer.train()
print(train_result)


In [ ]:
# Cell 14
# ======================================================
# Plot Training Loss Over Time
log_history = trainer.state.log_history
train_logs  = [e for e in log_history if "loss" in e]

steps  = [e["step"] for e in train_logs]
losses = [e["loss"] for e in train_logs]

plt.figure(figsize=(10, 4))
plt.plot(steps, losses, linewidth=2)
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("Training Loss Over Steps : RoBERTa Full Dataset")
plt.grid(axis="y", linestyle="--", linewidth=0.7)
plt.tight_layout()
plt.savefig("training_loss_curve.png", bbox_inches="tight")
plt.show()



In [ ]:
# Cell 15
# ======================================================
# Plot Macro and Micro F1 Across Epochs

eval_logs = [e for e in log_history if "eval_macro_f1" in e]

epochs    = [i + 1 for i in range(len(eval_logs))]
macro_f1s = [e["eval_macro_f1"] for e in eval_logs]
micro_f1s = [e["eval_micro_f1"] for e in eval_logs]

plt.figure(figsize=(8, 4))
plt.plot(epochs, macro_f1s, marker="o", linewidth=2, label="Macro F1 (class-balanced)")
plt.plot(epochs, micro_f1s, marker="s", linewidth=2, label="Micro F1 (overall)")
plt.xticks(epochs)
plt.xlabel("Epoch")
plt.ylabel("F1 Score")
plt.title("Macro and Micro F1 Across Training Epochs : Dev Set")
plt.legend()
plt.grid(axis="y", linestyle="--", linewidth=0.7)
plt.tight_layout()
plt.savefig("f1_across_epochs.png", bbox_inches="tight")
plt.show()



In [ ]:
# Cell 16
# ======================================================
# Save the Trained Model

from google.colab import drive, files
import shutil

# Save to Colab local storage
LOCAL_PATH = "roberta_final_model"
trainer.save_model(LOCAL_PATH)
tokenizer.save_pretrained(LOCAL_PATH)
print("Model saved to Colab local storage.")

# Zip and download to laptop



In [ ]:
# Cell 17
# ======================================================
# Run Final Evaluation on the Test Set

raw_preds   = trainer.predict(test_ds)
logits      = raw_preds.predictions
labels      = raw_preds.label_ids
probs       = torch.sigmoid(torch.tensor(logits)).numpy()
predictions = (probs >= 0.3).astype(int)

print("Test samples evaluated:", len(predictions))


In [ ]:
# Cell 18
# ======================================================
# Classification Report
from sklearn.metrics import classification_report
report = classification_report(
    labels,
    predictions,
    target_names=emotions,
    zero_division=0
)

print("Final Model : Classification Report (Test Set)")
print(report)

In [ ]:
# Cell 19
# ======================================================
# Per-Class F1 Score Chart

per_class_f1    = f1_score(labels, predictions, average=None, zero_division=0)
sorted_idx      = np.argsort(per_class_f1)
sorted_emotions = [emotions[i] for i in sorted_idx]
sorted_f1       = per_class_f1[sorted_idx]

plt.figure(figsize=(10, 12))
bars = plt.barh(sorted_emotions, sorted_f1, edgecolor="white")

for bar, val in zip(bars, sorted_f1):
    plt.text(
        val + 0.005,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.3f}",
        va="center", ha="left", fontsize=8
    )

plt.xlabel("F1 Score")
plt.ylabel("Emotion")
plt.title("Per-Class F1 Score : RoBERTa Final Model (Test Set)")
plt.xlim(0, 1.1)
plt.grid(axis="x", linestyle="--", linewidth=0.7)
plt.tight_layout()
plt.savefig("per_class_f1.png", bbox_inches="tight")
plt.show()


In [ ]:
# Cell 20
# ======================================================
# True Positive Rate per Emotion Class (Heatmap)

tp_rates = []
for i in range(NUM_EMOTIONS):
    true = labels[:, i]
    pred = predictions[:, i]
    cm   = confusion_matrix(true, pred, labels=[0, 1])
    if cm.shape == (2, 2) and (cm[1, 0] + cm[1, 1]) > 0:
        tpr = cm[1, 1] / (cm[1, 0] + cm[1, 1])
    else:
        tpr = 0.0
    tp_rates.append(round(tpr, 3))

tpr_df = pd.DataFrame({
    "Emotion":            emotions,
    "True Positive Rate": tp_rates
}).sort_values("True Positive Rate", ascending=False)

plt.figure(figsize=(8, 10))
sns.heatmap(
    tpr_df.set_index("Emotion"),
    annot=True,
    fmt=".3f",
    cmap="Blues",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "True Positive Rate"}
)
plt.title("True Positive Rate per Emotion : Final Model (Test Set)")
plt.tight_layout()
plt.savefig("confusion_matrix.png", bbox_inches="tight")
plt.show()



In [ ]:
# Cell 21
# ======================================================
# Final Performance Summary

final_macro    = f1_score(labels, predictions, average="macro",    zero_division=0)
final_micro    = f1_score(labels, predictions, average="micro",    zero_division=0)
final_weighted = f1_score(labels, predictions, average="weighted", zero_division=0)


print("Final Model : Performance Summary (Test Set)")
print(f"Macro F1    : {final_macro:.4f}")
print(f"Micro F1    : {final_micro:.4f}")
print(f"Weighted F1 : {final_weighted:.4f}")


## Survey Data Integration

This section loads responses from both surveys and runs them through the trained model.

**Survey 1 (Psychology Students)** : used for qualitative analysis:
comfort scores, feature preferences, and opinions on the AI Diary concept.

**Survey 2 (General Participants)** : used for real world model validation:
participants wrote freely about how they felt, then selected their emotions.
The model predicts emotions from the text and we compare against what they reported.


In [ ]:
# Cell 22
# ======================================================
# Upload Survey1 and Survey2
from google.colab import files

print("Upload survey1.csv")
s1 = files.upload()
survey_keys = [list(s1.keys())[0]]

print("Upload survey2.csv")
s2 = files.upload()
survey_keys.append(list(s2.keys())[0])

print("Files loaded:", survey_keys)

In [ ]:
# Cell 23
# ======================================================
# Load and Clean Survey 1

survey1_raw = pd.read_csv(survey_keys[0])
survey1_raw.columns = [
    "timestamp", "consent", "year_of_study",
    "heard_of_ai", "comfort_score", "useful",
    "benefits", "risks", "who_benefits",
    "features", "privacy_importance", "tone_preference",
    "would_use", "final_thoughts"
]

# Keep only responses where the participant gave consent
survey1 = survey1_raw[
    survey1_raw["consent"].str.strip().str.lower().str.startswith("yes")
].copy().reset_index(drop=True)

print("Survey 1 : Psychology Students")
print("  Total responses    :", len(survey1_raw))
print("  Consented responses:", len(survey1))
print()
print(survey1.head(3))


In [ ]:
# Cell 24
# ======================================================
# Load and Clean Survey 2

survey2_raw = pd.read_csv(survey_keys[1])
survey2_raw.columns = [
    "timestamp", "consent", "age", "gender", "situation",
    "feelings_text", "event_text", "emotions_selected",
    "intensity", "coping_methods", "preferred_response",
    "ai_vs_person", "resources", "final_thoughts"
]

# Keep only responses where the participant gave consent
survey2 = survey2_raw[
    survey2_raw["consent"].str.strip().str.lower().str.startswith("yes")
].copy().reset_index(drop=True)

print("Survey 2 : General Participants")
print("  Total responses    :", len(survey2_raw))
print("  Consented responses:", len(survey2))
print()
print(survey2.head(3))


In [ ]:
# Cell 25
# ======================================================
# Survey 1 : Comfort with AI Chart

comfort_counts = survey1["comfort_score"].value_counts().sort_index()
plt.figure(figsize=(8, 4))
bars = plt.bar(
    comfort_counts.index.astype(str),
    comfort_counts.values,
    edgecolor="white"
)

for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.1,
        str(int(bar.get_height())),
        ha="center", va="bottom", fontsize=10
    )

plt.xlabel("Comfort Score (1 = Not comfortable, 5 = Very comfortable)")
plt.ylabel("Number of Responses")
plt.title("Psychology Students : Comfort with AI for Emotional Reflection")
plt.grid(axis="y", linestyle="--", linewidth=0.7)
plt.tight_layout()
plt.savefig("survey1_comfort.png", bbox_inches="tight")
plt.show()



In [ ]:
# Cell 26
# ======================================================
# Survey 1 : Feature Preference Chart

all_features = []
for row in survey1["features"].dropna():
    all_features.extend([item.strip() for item in str(row).split(";")])

feature_counts = Counter(all_features)
feature_df = pd.DataFrame(
    feature_counts.items(), columns=["Feature", "Count"]
).sort_values("Count", ascending=True)

plt.figure(figsize=(10, 5))
plt.barh(feature_df["Feature"], feature_df["Count"], edgecolor="white")
plt.xlabel("Number of Responses")
plt.ylabel("Feature")
plt.title("Psychology Students : Most Useful AI Diary Features")
plt.grid(axis="x", linestyle="--", linewidth=0.7)
plt.tight_layout()
plt.savefig("survey1_features.png", bbox_inches="tight")
plt.show()



In [ ]:
# Cell 27
# ======================================================
# Reload the Model for Running on Survey Text

from google.colab import drive
drive.mount('/content/drive')

SAVE_PATH = "/content/drive/MyDrive/AI_Diary/roberta_final_model"

inference_tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)
inference_model     = AutoModelForSequenceClassification.from_pretrained(SAVE_PATH)
inference_model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inference_model.to(device)

print("Model reloaded from:", SAVE_PATH)
print("Running on device  :", device)

In [ ]:
# Cell 28
# ======================================================
# Emotion Prediction Function


def predict_emotions(text, threshold=0.3):
    inputs = inference_tokenizer(
        str(text),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        logits = inference_model(**inputs).logits

    probs     = torch.sigmoid(logits).squeeze().cpu().numpy()
    predicted = [emotions[i] for i, p in enumerate(probs) if p >= threshold]

    return predicted, probs


# example test
test_text   = "I have not been feeling okay because i failed my exam ."
test_result = predict_emotions(test_text)
print("Test prediction:")
print("  Text     :", test_text)
print("  Emotions :", test_result[0])


In [ ]:
# Cell 29
# ======================================================
# Run the Model on Survey 2 Participant Response.

def combine_text(row):
    parts = []
    if pd.notna(row["feelings_text"]) and str(row["feelings_text"]).strip():
        parts.append(str(row["feelings_text"]).strip())
    if pd.notna(row["event_text"]) and str(row["event_text"]).strip():
        parts.append(str(row["event_text"]).strip())
    return " ".join(parts)

survey2["combined_text"] = survey2.apply(combine_text, axis=1)

survey2["model_predictions"] = survey2["combined_text"].apply(
    lambda x: predict_emotions(x)[0] if str(x).strip() else []
)

print("Predictions generated for", len(survey2), "responses.")
print()
print("First 5 examples:")
for i, row in survey2[["combined_text", "model_predictions"]].head(5).iterrows():
    print(f"  [{i+1}] Text     : {str(row['combined_text'])[:80]}...")
    print(f"      Predicted: {row['model_predictions']}")
    print()


In [ ]:
# Cell 30
# ======================================================
# Map Survey Emotion Labels to GoEmotions Labels

emotion_mapping = {
    "Anxious":         "nervousness",
    "Calm":            "relief",
    "Sad":             "sadness",
    "Angry":           "anger",
    "Happy":           "joy",
    "Lonely":          "sadness",
    "Stressed":        "nervousness",
    "Overwhelmed":     "nervousness",
    "Hopeful":         "optimism",
    "Numb / Not sure": "neutral"
}


def parse_human_labels(label_string):
    if pd.isna(label_string):
        return []
    raw = [item.strip() for item in str(label_string).split(";")]
    return [emotion_mapping.get(l, l) for l in raw if l]


survey2["human_labels"] = survey2["emotions_selected"].apply(parse_human_labels)

print("Human labels mapped to GoEmotions labels.")
print()
print(survey2[["emotions_selected", "human_labels"]].head(5))


In [ ]:
# Cell 31
# ======================================================
# Agreement Score : Model Predictions vs Human Labels


def agreement_score(human, predicted):
    if not human or not predicted:
        return None
    return len(set(human) & set(predicted)) / len(set(human))

survey2["agreement"] = survey2.apply(
    lambda row: agreement_score(row["human_labels"], row["model_predictions"]),
    axis=1
)

valid = survey2["agreement"].dropna()


print("Model vs Human Agreement Summary")

print("Responses with a score :", len(valid))
print("Mean agreement         :", round(valid.mean(), 4))
print("Median agreement       :", round(valid.median(), 4))
print("Std deviation          :", round(valid.std(), 4))
print("Perfect agreement (1.0):", (valid == 1.0).sum())
print("No agreement at all    :", (valid == 0.0).sum())


In [ ]:
# Cell 32
# ======================================================
# Model vs Human Agreement Chart

plt.figure(figsize=(9, 4))
plt.hist(valid, bins=10, edgecolor="white")
plt.axvline(
    valid.mean(),
    linestyle="--", linewidth=1.5,
    label=f"Mean: {valid.mean():.2f}"
)
plt.xlabel("Agreement Score (0 = No overlap, 1 = Full overlap)")
plt.ylabel("Number of Participants")
plt.title("Model vs Human Label Agreement : Survey 2 Participants")
plt.legend()
plt.grid(axis="y", linestyle="--", linewidth=0.7)
plt.tight_layout()
plt.savefig("agreement_chart.png", bbox_inches="tight")
plt.show()



In [ ]:
# Cell 33
# ======================================================
# Self reported emotion frequency


all_reported = []
for row in survey2["emotions_selected"].dropna():
    all_reported.extend([item.strip() for item in str(row).split(";")])

emo_freq = Counter(all_reported)
emo_df   = pd.DataFrame(
    emo_freq.items(), columns=["Emotion", "Count"]
).sort_values("Count", ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(emo_df["Emotion"], emo_df["Count"], edgecolor="white")
plt.xlabel("Number of Participants")
plt.ylabel("Emotion")
plt.title("Self-Reported Emotions : Survey 2 Participants")
plt.grid(axis="x", linestyle="--", linewidth=0.7)
plt.tight_layout()
plt.savefig("survey2_emotions.png", bbox_inches="tight")
plt.show()



In [ ]:
# Cell 34
# ======================================================
# Survey 2 : Preferred AI Response Style

response_counts = survey2["preferred_response"].value_counts()

plt.figure(figsize=(10, 4))
bars = plt.barh(response_counts.index, response_counts.values, edgecolor="white")

for bar in bars:
    plt.text(
        bar.get_width() + 0.1,
        bar.get_y() + bar.get_height() / 2,
        str(int(bar.get_width())),
        va="center", ha="left", fontsize=10
    )

plt.xlabel("Number of Participants")
plt.ylabel("Response Type")
plt.title("Preferred AI Response Style : Survey 2 Participants")
plt.grid(axis="x", linestyle="--", linewidth=0.7)
plt.tight_layout()
plt.savefig("survey2_preferred_response.png", bbox_inches="tight")
plt.show()



## Reflection System

This section generates short, empathetic responses based on the emotions the model detects.


In [ ]:
# Cell 35
# ======================================================
# Reflection Template Dictionary

TEMPLATES = {
    "sadness":        "It sounds like things have been quite heavy recently. That is completely okay, sometimes sitting with difficult feelings is part of processing them.",
    "joy":            "It is lovely to hear something positive has been happening for you. Holding onto moments like that can make a real difference.",
    "anger":          "It makes sense that you are feeling frustrated. Your feelings are valid, and it can help to take a moment before responding to whatever caused this.",
    "fear":           "Feeling afraid or unsettled can be really draining. You do not have to face what is worrying you all at once.",
    "nervousness":    "It sounds like you are carrying quite a bit of tension. Some people find it helpful to step away for a short break or take a few slow breaths.",
    "disgust":        "It is understandable to feel uncomfortable by something. Giving yourself some space from the situation can sometimes help.",
    "surprise":       "It sounds like something unexpected has come your way. It is natural to need a little time to process something you were not expecting.",
    "anticipation":   "It sounds like there is something ahead that has your attention. Whether it is exciting or nerve-wracking, that kind of anticipation is completely normal.",
    "love":           "It is clear that something or someone matters a great deal to you. That kind of connection is really meaningful.",
    "gratitude":      "It is positive that you are noticing things to be grateful for. That kind of awareness can be a real source of strength.",
    "optimism":       "It is good to hear that you are feeling hopeful. Holding onto that sense of possibility is worth nurturing.",
    "caring":         "The care you have for others comes through clearly. Just make sure you are giving some of that care to yourself too.",
    "excitement":     "That energy comes through in what you have written. It is great to have something to look forward to.",
    "amusement":      "It sounds like something brought a smile today. Those lighter moments are worth holding onto.",
    "admiration":     "It is clear that something or someone has genuinely impressed you. That kind of appreciation is a positive thing to notice.",
    "approval":       "It sounds like you are feeling good about how something has gone. Recognising what is working well is important.",
    "curiosity":      "Your curiosity and openness comes through in how you have written. That kind of inquisitiveness is a real strength.",
    "confusion":      "It sounds like things are feeling a bit unclear right now. It is okay not to have all the answers immediately.",
    "embarrassment":  "Feeling embarrassed is something everyone goes through, even if it does not always feel that way. It tends to feel bigger in the moment than it really is.",
    "disappointment": "It sounds like something has not gone the way you hoped. Disappointment is a sign that you cared about the outcome, which is not a bad thing.",
    "grief":          "What you are going through sounds genuinely painful. Grief takes time and does not follow a set schedule. Please be patient with yourself.",
    "remorse":        "It sounds like something is sitting heavily with you. Acknowledging how you feel is already a meaningful first step.",
    "relief":         "It is clear that a weight has been lifted. Allow yourself to enjoy that feeling, you deserve it.",
    "pride":          "It is clear that you have achieved something that matters to you. Taking a moment to acknowledge that is completely deserved.",
    "realization":    "It sounds like something has clicked for you recently. Those moments of clarity, even when difficult, can be really important.",
    "desire":         "It sounds like there is something you are working towards. Knowing what you want is a meaningful starting point.",
    "annoyance":      "It sounds like something has been getting under your skin. Those feelings are valid, small frustrations can build up over time.",
    "neutral":        "Thank you for taking the time to write today. Even on quieter days, reflection has value."
}

DEFAULT = "Thank you for sharing how you are feeling. Whatever you are going through, writing it down is a positive step."

print("Templates loaded for", len(TEMPLATES), "emotions.")


In [ ]:
# Cell 36
# ======================================================
# Reflection Generation Function

def generate_reflection(text, threshold=0.3):
    predicted, probs = predict_emotions(text, threshold)

    if not predicted:
        return {
            "input_text":        text,
            "detected_emotions": [],
            "primary_emotion":   "none detected",
            "reflection":        DEFAULT
        }

    top_emotion = emotions[int(np.argmax(probs))]
    reflection  = TEMPLATES.get(top_emotion, DEFAULT)

    return {
        "input_text":        text,
        "detected_emotions": predicted,
        "primary_emotion":   top_emotion,
        "reflection":        reflection
    }


In [ ]:
# Cell 37
# ======================================================
# Reflection System Demo


demo_texts = [
    "I have been feeling really low this week. Nothing seems to be going right and I just feel exhausted all the time.",
    "I finally got the news I had been waiting for and I am so relieved. I honestly did not think it would work out.",
    "I am not sure how I feel today. Things are fine I suppose but I feel a bit flat and disconnected from everything."
]


print("AI Diary : Reflection System Demo")
print()

for i, text in enumerate(demo_texts, 1):
    result = generate_reflection(text)
    print(f"Example {i}")
    print("Input     :", result["input_text"])
    print("Detected  :", result["detected_emotions"])
    print("Primary   :", result["primary_emotion"])
    print("Reflection:", result["reflection"])
    print()


In [ ]:
# Cell 38
# ======================================================
# Run Reflection System on All Survey 2 Responses

results = survey2["combined_text"].apply(
    lambda x: generate_reflection(str(x)) if str(x).strip() else None
)

survey2["primary_emotion"] = results.apply(lambda r: r["primary_emotion"] if r else None)
survey2["reflection"]      = results.apply(lambda r: r["reflection"]      if r else None)

output_cols = [
    "combined_text", "emotions_selected",
    "model_predictions", "human_labels",
    "agreement", "primary_emotion", "reflection"
]

survey2[output_cols].to_csv("survey2_results.csv", index=False)

print("survey2_results.csv")
print()
print(survey2[output_cols].head(5))


In [ ]:
# Cell 39
# ======================================================
# Summary

print("=" * 50)
print("AI Diary : Notebook 03 Complete")
print("=" * 50)

print("\nModel performance on test set:")
print("  Macro F1    :", round(final_macro, 4))
print("  Micro F1    :", round(final_micro, 4))
print("  Weighted F1 :", round(final_weighted, 4))
print("  Test samples:", len(predictions))

print("\nTraining configuration:")
print("  Model       : roberta-base")
print("  Epochs      : 3")
print("  Batch size  : 8")
print("  Threshold   : 0.3")
print("  Class weighting : Yes")

print("\nSurvey 2 validation:")
print("  Participants scored :", len(valid))
print("  Mean agreement      :", round(valid.mean(), 4))
print("  Median agreement    :", round(valid.median(), 4))
print("  Perfect agreement   :", (valid == 1.0).sum())
print("  Zero agreement      :", (valid == 0.0).sum())

print("\nBaseline comparison:")
print("  ClinicalBERT Macro F1  : 0.0220")
print("  RoBERTa prototype Macro F1 : 0.0178")
print("  Final model Macro F1   :", round(final_macro, 4))
print("=" * 50)